# SVAMP — GuidedVote Floating-Point Precision Evaluation
**Research Question**: Does quantizing the guide model degrade GuidedVote accuracy?

**Design**:
- Solver is **fixed at FP16** throughout (same as paper)
- Guide precision is varied: `BF16 (reference) → FP16 → INT8 → INT4`
- Baseline (no guide) is run **once** — it does not depend on guide precision
- 150 questions, seed 42 — same split for all precision conditions
- Three angles reported per precision: Accuracy, Vote Consistency, ECE

**Output**: `precision_results/` folder with per-precision JSONL + final comparison table

In [ ]:
# CELL 1 -- Install (uncomment on first run)
# !pip install -q transformers==4.44.0
# !pip install -q peft==0.12.0
# !pip install -q accelerate==0.33.0
# !pip install -q datasets==2.20.0
# !pip install -q bitsandbytes>=0.43.0
# !pip install -q huggingface_hub
print("Done.")

In [ ]:
# CELL 2 -- HuggingFace login
from huggingface_hub import login
login("")
print('✅ HuggingFace login done')

In [ ]:
# CELL 3 -- Imports + GPU
import os, json, re, glob, random, time, gc
import torch
import numpy as np
from collections import Counter
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
from tqdm.notebook import tqdm

OUTPUT_DIR = "/kaggle/working/svamp_precision_eval"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"PyTorch      : {torch.__version__}")
print(f"GPU          : {torch.cuda.get_device_name(0)}")
print(f"VRAM         : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"Output dir   : {OUTPUT_DIR}")

In [ ]:
# CELL 4 -- Configuration
CONFIG = {
    # Models
    "guide_base"          : "Qwen/Qwen2.5-3B-Instruct",
    "response_model"      : "Qwen/Qwen2.5-1.5B-Instruct",

    # Dataset
    "dataset_name"        : "ChilleD/SVAMP",
    "dataset_split"       : "test",
    "max_eval_samples"    : 150,   # Reduced from paper — sufficient for precision study
    "random_seed"         : 42,

    # Ensemble
    "n_votes"             : 5,
    "vote_temperature"    : 0.7,
    "guide_temperature"   : 0.1,
    "refiner_temperature" : 0.3,
    "max_new_tokens"      : 350,

    # Compute cost (billions of parameters)
    "guide_params_B"      : 3.0,
    "solver_params_B"     : 1.5,

    # Precision conditions to evaluate (in this order)
    # bf16 = reference (matches your paper), then decreasing precision
    "precisions"          : ["bf16", "fp16", "int8", "int4"],

    # Checkpoint every N questions within a precision run
    "save_every"          : 25,
}

print("Config ready:")
for k, v in CONFIG.items():
    print(f"  {k:<24}: {v}")

In [ ]:
# CELL 5 -- Load SVAMP dataset (same as original notebook)

print("Loading SVAMP from HuggingFace...")
raw_ds = load_dataset(CONFIG["dataset_name"])

print(f"Splits   : {list(raw_ds.keys())}")
print(f"Test size: {len(raw_ds[CONFIG['dataset_split']])}")


def normalise_svamp(item):
    """Convert SVAMP record to {question, answer}."""
    q = item["Body"].strip().rstrip(".") + " " + item["Question"].strip()
    ans = item["Answer"]
    if isinstance(ans, float) and ans == int(ans):
        ans_str = str(int(ans))
    else:
        ans_str = str(ans)
    return {"question": q, "answer": ans_str}


all_data = [normalise_svamp(x) for x in raw_ds[CONFIG["dataset_split"]]]

# CRITICAL: seed fixed once — same questions for ALL precision conditions
random.seed(CONFIG["random_seed"])
test_data = random.sample(all_data, CONFIG["max_eval_samples"])

print(f"\nSampled {len(test_data)} questions (seed={CONFIG['random_seed']})")
print(f"First Q : {test_data[0]['question'][:80]}")
print(f"First A : {test_data[0]['answer']}")
print("SVAMP loaded — same split will be used for all precision conditions")

In [ ]:
# CELL 6 -- Answer extraction (same as original notebook)

def normalise_num(s):
    s = s.replace(",", "").strip()
    try:
        f = float(s)
        return str(int(f)) if f == int(f) else str(round(f, 4))
    except ValueError:
        return s


def extract_gt_answer(answer_str):
    return normalise_num(str(answer_str))


def extract_pred_answer(text):
    m = re.search(r"####\s*(-?[\d\.]+)", text)
    if m: return normalise_num(m.group(1))
    m = re.search(r"\\boxed\{(-?[\d\.]+)\}", text)
    if m: return normalise_num(m.group(1))
    m = re.search(r"(?:the answer is|answer is)\s*:?\s*\$?(-?[\d\.]+)", text, re.IGNORECASE)
    if m: return normalise_num(m.group(1))
    m = re.search(r"=\s*\$?(-?[\d\.]+)\s*$", text.strip(), re.MULTILINE)
    if m: return normalise_num(m.group(1))
    m = re.search(r"\*\*\$?(-?[\d\.]+)\*\*\.?\s*$", text.strip())
    if m: return normalise_num(m.group(1))
    m = re.search(r"(?:therefore|thus|so|hence)[,\s]+(?:the answer is\s*)?\$?(-?[\d\.]+)", text, re.IGNORECASE)
    if m: return normalise_num(m.group(1))
    return ""


# Self-test
_tests = [
    ("#### 42", "42"), ("#### 3.5", "3.5"), ("\\boxed{100}", "100"),
    ("The answer is 7", "7"), ("Total = 20", "20"),
    ("**200**.", "200"), ("Therefore, 13", "13"), ("Some unrelated text", ""),
]
ok = True
for txt, exp in _tests:
    got = extract_pred_answer(txt)
    status = "OK" if got == exp else "FAIL"
    if got != exp: ok = False
    print(f"  {status}  '{txt[:35]}' -> '{got}' (expected '{exp}')")
print("All extractor tests passed" if ok else "EXTRACTOR HAS FAILURES")

In [ ]:
# CELL 7 -- Load solver ONCE (FP16, fixed for the entire experiment)
#
# The solver never changes precision. Only the guide changes.
# This matches the paper's setup and isolates the precision effect on the guide.

print(f"Loading solver (fixed FP16): {CONFIG['response_model']}")
resp_tok = AutoTokenizer.from_pretrained(CONFIG["response_model"])
if resp_tok.pad_token is None:
    resp_tok.pad_token = resp_tok.eos_token

resp_model = AutoModelForCausalLM.from_pretrained(
    CONFIG["response_model"],
    torch_dtype=torch.float16,
    device_map="auto",
).eval()

solver_vram = torch.cuda.memory_allocated() / 1e9
print(f"Solver VRAM : {solver_vram:.2f} GB")
print(f"Headroom for guide : {17.1 - solver_vram:.1f} GB")
print("Solver loaded — will NOT be unloaded between precision runs")

In [ ]:
# CELL 8 -- Guide loader: loads guide in specified precision + LoRA adapter

def find_adapter():
    patterns = [
        "/kaggle/input/datasets/sufiantabdullah/final-adapter",
        "/kaggle/input/*/adapter",
        "/kaggle/input/*/final-adapter",
        "/kaggle/input/*/final_adapter",
    ]
    for p in patterns:
        for m in glob.glob(p):
            print(f"  Found adapter: {m}")
            return m
    return None


def load_guide(precision: str):
    """
    Load guide base model + LoRA adapter in the given precision.
    Returns (model, tokenizer).

    precision options: 'bf16', 'fp16', 'int8', 'int4'

    NOTE on INT8/INT4 + LoRA:
    BitsAndBytesConfig quantizes the base weights. PeftModel then adds
    the LoRA adapter layers on top in their original dtype (bf16/fp16).
    This is the standard approach and is fully supported.
    """
    print(f"\nLoading guide [{precision.upper()}]: {CONFIG['guide_base']}")

    tok = AutoTokenizer.from_pretrained(CONFIG["guide_base"], trust_remote_code=True)
    tok.padding_side = "left"
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token

    # Build load kwargs based on precision
    load_kwargs = {"device_map": "auto", "trust_remote_code": True}

    if precision == "bf16":
        load_kwargs["torch_dtype"] = torch.bfloat16

    elif precision == "fp16":
        load_kwargs["torch_dtype"] = torch.float16

    elif precision == "int8":
        bnb_config = BitsAndBytesConfig(load_in_8bit=True)
        load_kwargs["quantization_config"] = bnb_config
        load_kwargs["torch_dtype"] = torch.float16  # compute dtype for non-quantized layers

    elif precision == "int4":
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.bfloat16,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
        )
        load_kwargs["quantization_config"] = bnb_config
    else:
        raise ValueError(f"Unknown precision: {precision}")

    model = AutoModelForCausalLM.from_pretrained(CONFIG["guide_base"], **load_kwargs)

    # Load LoRA adapter
    adapter_path = find_adapter()
    if adapter_path:
        model = PeftModel.from_pretrained(model, adapter_path)
        print(f"  LoRA adapter loaded")
    else:
        print("  WARNING: No adapter found. Using base model as guide.")

    model.eval()
    guide_vram = torch.cuda.memory_allocated() / 1e9
    solver_vram_used = solver_vram  # from Cell 7
    print(f"  Guide VRAM  : {guide_vram - solver_vram:.2f} GB")
    print(f"  Total VRAM  : {guide_vram:.2f} GB / 17.1 GB")
    return model, tok


def unload_guide(model, tok):
    """Release guide from GPU memory."""
    del model, tok
    gc.collect()
    torch.cuda.empty_cache()
    print(f"  Guide unloaded. VRAM now: {torch.cuda.memory_allocated() / 1e9:.2f} GB")


print("Guide loader ready")

In [ ]:
# CELL 9 -- Prompts and generation functions
# Same as original notebook. guide_model / guide_tok are reassigned
# at the start of each precision loop iteration (Cell 12).

GUIDE_SYSTEM = (
    "You are a math problem decomposition assistant.\n"
    "Identify ONLY the arithmetic operations needed. 1-3 steps maximum.\n"
    "State WHAT is being compared or combined using EXACT numbers.\n"
    "Do NOT invent steps. Do NOT reorder the question.\n\n"
    "BAD:  Step 1: Calculate remaining cookies.\n"
    "GOOD: Step 1: Eaten - Given = 14 - 13 = ?\n"
    "      Step 2: Answer = 14 - 13\n\n"
    "If the question asks 'how many MORE did X than Y', "
    "the operation is X - Y, not Y - X.\n"
    "No final answer number. Just the operation steps."
)

SOLVE_SYSTEM = (
    "You are a math problem solver.\n"
    "Compute each step numerically. No markdown. No bullet points. No headers.\n"
    "Write plain arithmetic steps only.\n"
    "Your absolute last line must be: #### [number]\n"
    "NEVER write ### or ** in your response.\n\n"
    "Example:\n"
    "Eaten = 14. Given = 13.\n"
    "Difference = 14 - 13 = 1.\n"
    "#### 1"
)

BASELINE_SYSTEM = (
    "You are a precise math problem solver.\n"
    "Read the problem carefully. Solve step by step, showing every calculation.\n"
    "Your FINAL line must be exactly: #### [number]"
)

REFINER_SYSTEM = (
    "You are a careful math problem solver.\n"
    "Previous attempts on this problem gave different answers.\n"
    "Re-solve completely from scratch using a fresh approach.\n"
    "Show every arithmetic step.\n"
    "Your FINAL line must be exactly: #### [number]"
)


def run_model(mdl, tok, messages, max_tokens, temperature):
    """Generic generation call. Works for both guide and solver."""
    prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tok(prompt, return_tensors="pt", truncation=True, max_length=1024)
    device = next(mdl.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        out = mdl.generate(
            **inputs,
            max_new_tokens     = max_tokens,
            temperature        = max(temperature, 0.05),
            do_sample          = True,
            top_p              = 0.92,
            top_k              = 40,
            pad_token_id       = tok.eos_token_id,
            repetition_penalty = 1.15,
        )
    new_toks = out[0][inputs["input_ids"].shape[1]:]
    return tok.decode(new_toks, skip_special_tokens=True).strip()


# These functions reference guide_model / guide_tok as globals.
# Cell 12 reassigns these globals before each precision run.

def generate_plan(question):
    return run_model(
        guide_model, guide_tok,
        [{"role": "system", "content": GUIDE_SYSTEM},
         {"role": "user",   "content": f"Problem: {question}"}],
        max_tokens  = 350,
        temperature = CONFIG["guide_temperature"],
    )


def generate_guided(question, plan):
    content = f"Problem: {question}\n\nPlan (follow each step):\n{plan}\n\nSolve step by step:"
    return run_model(
        resp_model, resp_tok,
        [{"role": "system", "content": SOLVE_SYSTEM},
         {"role": "user",   "content": content}],
        max_tokens  = CONFIG["max_new_tokens"],
        temperature = CONFIG["vote_temperature"],
    )


def generate_baseline(question):
    return run_model(
        resp_model, resp_tok,
        [{"role": "system", "content": BASELINE_SYSTEM},
         {"role": "user",   "content": f"Problem: {question}"}],
        max_tokens  = CONFIG["max_new_tokens"],
        temperature = CONFIG["vote_temperature"],
    )


def generate_refiner(question, plan, candidates):
    cands = ", ".join(sorted(set(c for c in candidates if c)))
    content = (
        f"Problem: {question}\n\n"
        f"Plan:\n{plan}\n\n"
        f"Previous attempts disagreed: {cands}\n"
        "Re-solve carefully from scratch:"
    )
    return run_model(
        resp_model, resp_tok,
        [{"role": "system", "content": REFINER_SYSTEM},
         {"role": "user",   "content": content}],
        max_tokens  = CONFIG["max_new_tokens"],
        temperature = CONFIG["refiner_temperature"],
    )


print("Generation functions ready")

In [ ]:
# CELL 10 -- Voting logic (same as original notebook)

def vote_and_decide(answers, question, plan, gt_answer=None):
    valid = [a for a in answers if a and a.strip()]
    if not valid:
        valid = answers
    vote_counts = Counter(valid)
    most_common = vote_counts.most_common()
    top_answer  = most_common[0][0]
    top_count   = most_common[0][1]
    total       = len(answers)

    correct_votes    = vote_counts.get(gt_answer, 0) if gt_answer else 0
    vote_consistency = correct_votes / total
    is_majority      = (len(most_common) == 1 or top_count > most_common[1][1])

    refiner_used = False
    refiner_correct = None

    if is_majority:
        final    = top_answer
        strategy = "majority"
        conf     = round(top_count / total, 4)
        wasted   = total - top_count
    else:
        ref_raw         = generate_refiner(question, plan, list(answers))
        ref_ans         = extract_pred_answer(ref_raw)
        refiner_used    = True
        refiner_correct = (ref_ans == gt_answer) if gt_answer else None

        all_votes  = answers + [ref_ans]
        new_counts = Counter(all_votes)
        new_common = new_counts.most_common()
        new_top    = new_common[0][0]
        new_top_c  = new_common[0][1]
        still_tied = len(new_common) > 1 and new_top_c == new_common[1][1]

        final            = new_top
        strategy         = "coin_flip" if still_tied else "refiner_tiebreak"
        conf             = round(new_top_c / len(all_votes), 4)
        vote_counts      = new_counts
        total            = len(all_votes)
        correct_votes    = new_counts.get(gt_answer, 0) if gt_answer else 0
        vote_consistency = correct_votes / total
        wasted           = total - new_top_c

    return {
        "final_answer"     : final,
        "strategy"         : strategy,
        "confidence"       : conf,
        "vote_counts"      : dict(vote_counts),
        "correct_votes"    : correct_votes,
        "total_votes"      : total,
        "vote_consistency" : round(vote_consistency, 4),
        "wasted_votes"     : wasted,
        "refiner_used"     : refiner_used,
        "refiner_correct"  : refiner_correct,
    }


print("Voting logic ready")

In [ ]:
# CELL 11 -- Run BASELINE once (solver only, no guide)
#
# Baseline does not depend on guide precision, so we run it ONCE
# here and reuse the results for all precision comparisons.

BASELINE_FILE = f"{OUTPUT_DIR}/baseline_results.jsonl"
BASELINE_CKPT = f"{OUTPUT_DIR}/baseline_checkpoint.json"

base_results = []
start_idx    = 0

if os.path.exists(BASELINE_CKPT):
    with open(BASELINE_CKPT) as f:
        ckpt = json.load(f)
    start_idx = ckpt.get("last_index", 0)
    if os.path.exists(BASELINE_FILE):
        with open(BASELINE_FILE) as f:
            base_results = [json.loads(l) for l in f if l.strip()]
    print(f"Resuming baseline from index {start_idx} ({len(base_results)} saved)")
else:
    print(f"Running baseline on {len(test_data)} questions...")

t0 = time.time()

for idx in tqdm(range(start_idx, len(test_data)), desc="Baseline"):
    item      = test_data[idx]
    question  = item["question"]
    gt_answer = extract_gt_answer(item["answer"])

    try:
        b_votes = [extract_pred_answer(generate_baseline(question))
                   for _ in range(CONFIG["n_votes"])]
        b_dec   = vote_and_decide(b_votes, question, "baseline", gt_answer)
        base_results.append({
            "idx": idx, "question": question, "gt_answer": gt_answer,
            "final_answer": b_dec["final_answer"],
            "correct": b_dec["final_answer"] == gt_answer,
            "confidence": b_dec["confidence"],
            "correct_votes": b_dec["correct_votes"],
            "total_votes": b_dec["total_votes"],
            "vote_consistency": b_dec["vote_consistency"],
            "wasted_votes": b_dec["wasted_votes"],
            "strategy": b_dec["strategy"],
        })
    except RuntimeError as e:
        base_results.append({
            "idx": idx, "question": question, "gt_answer": gt_answer,
            "final_answer": "", "correct": False, "confidence": 0.0,
            "correct_votes": 0, "total_votes": CONFIG["n_votes"],
            "vote_consistency": 0.0, "wasted_votes": CONFIG["n_votes"],
            "strategy": "error", "error": str(e),
        })

    if (idx + 1) % CONFIG["save_every"] == 0:
        with open(BASELINE_FILE, "w") as f:
            for r in base_results:
                f.write(json.dumps(r) + "\n")
        with open(BASELINE_CKPT, "w") as f:
            json.dump({"last_index": idx + 1}, f)
        b_acc = sum(r["correct"] for r in base_results) / len(base_results) * 100
        mins  = (time.time() - t0) / 60
        print(f"  [{idx+1:3d}] Baseline: {b_acc:.1f}%  ({mins:.1f} min)")

# Final save
with open(BASELINE_FILE, "w") as f:
    for r in base_results:
        f.write(json.dumps(r) + "\n")

b_final_acc = sum(r["correct"] for r in base_results) / len(base_results) * 100
print(f"\nBaseline complete: {b_final_acc:.1f}% ({sum(r['correct'] for r in base_results)}/{len(base_results)})")
print(f"Time: {(time.time()-t0)/60:.1f} min")

In [ ]:
# CELL 12 -- Main Precision Loop
#
# For each precision in [bf16, fp16, int8, int4]:
#   1. Load guide in that precision
#   2. Run all 150 questions (guided only — baseline already done)
#   3. Save results to {precision}_guided_results.jsonl
#   4. Unload guide, clear VRAM
#
# Checkpoint: saves every save_every questions so a crash does not
# lose progress within a precision run.
# If a precision run is already complete (file exists with 150 records),
# it is skipped automatically.

precision_results = {}   # precision -> list of result dicts

for precision in CONFIG["precisions"]:
    results_file = f"{OUTPUT_DIR}/{precision}_guided_results.jsonl"
    ckpt_file    = f"{OUTPUT_DIR}/{precision}_checkpoint.json"

    # --- Skip if already complete ---
    if os.path.exists(results_file):
        with open(results_file) as f:
            existing = [json.loads(l) for l in f if l.strip()]
        if len(existing) >= len(test_data):
            print(f"\n[{precision.upper()}] Already complete ({len(existing)} records). Skipping.")
            precision_results[precision] = existing
            continue

    print(f"\n{'='*65}")
    print(f"PRECISION RUN: {precision.upper()}")
    print(f"{'='*65}")

    # --- Load guide for this precision ---
    guide_model, guide_tok = load_guide(precision)

    # --- Resume from checkpoint if exists ---
    guided_results = []
    start_idx = 0
    if os.path.exists(ckpt_file):
        with open(ckpt_file) as f:
            ckpt = json.load(f)
        start_idx = ckpt.get("last_index", 0)
        if os.path.exists(results_file):
            with open(results_file) as f:
                guided_results = [json.loads(l) for l in f if l.strip()]
        print(f"  Resuming from index {start_idx} ({len(guided_results)} saved)")

    # --- Evaluation loop ---
    t0 = time.time()
    for idx in tqdm(range(start_idx, len(test_data)), desc=f"{precision.upper()}"):
        item      = test_data[idx]
        question  = item["question"]
        gt_answer = extract_gt_answer(item["answer"])

        try:
            t_q = time.time()
            plan    = generate_plan(question)
            g_votes = [extract_pred_answer(generate_guided(question, plan))
                       for _ in range(CONFIG["n_votes"])]
            g_dec   = vote_and_decide(g_votes, question, plan, gt_answer)
            q_time  = time.time() - t_q

            guided_results.append({
                "precision"       : precision,
                "idx"             : idx,
                "question"        : question,
                "gt_answer"       : gt_answer,
                "final_answer"    : g_dec["final_answer"],
                "correct"         : g_dec["final_answer"] == gt_answer,
                "confidence"      : g_dec["confidence"],
                "correct_votes"   : g_dec["correct_votes"],
                "total_votes"     : g_dec["total_votes"],
                "vote_consistency": g_dec["vote_consistency"],
                "wasted_votes"    : g_dec["wasted_votes"],
                "strategy"        : g_dec["strategy"],
                "refiner_used"    : g_dec["refiner_used"],
                "refiner_correct" : g_dec["refiner_correct"],
                "plan"            : plan,
                "inference_time"  : round(q_time, 3),
            })
        except RuntimeError as e:
            guided_results.append({
                "precision": precision, "idx": idx, "question": question,
                "gt_answer": gt_answer, "final_answer": "", "correct": False,
                "confidence": 0.0, "correct_votes": 0,
                "total_votes": CONFIG["n_votes"], "vote_consistency": 0.0,
                "wasted_votes": CONFIG["n_votes"], "strategy": "error",
                "refiner_used": False, "refiner_correct": None,
                "plan": "", "inference_time": 0.0, "error": str(e),
            })

        # Checkpoint
        if (idx + 1) % CONFIG["save_every"] == 0:
            with open(results_file, "w") as f:
                for r in guided_results:
                    f.write(json.dumps(r) + "\n")
            with open(ckpt_file, "w") as f:
                json.dump({"last_index": idx + 1}, f)
            g_acc = sum(r["correct"] for r in guided_results) / len(guided_results) * 100
            mins  = (time.time() - t0) / 60
            print(f"  [{idx+1:3d}] {precision.upper()}: {g_acc:.1f}%  ({mins:.1f} min)")

    # Final save for this precision
    with open(results_file, "w") as f:
        for r in guided_results:
            f.write(json.dumps(r) + "\n")

    g_acc = sum(r["correct"] for r in guided_results) / len(guided_results) * 100
    print(f"\n  [{precision.upper()}] Done: {g_acc:.1f}%  Time: {(time.time()-t0)/60:.1f} min")

    precision_results[precision] = guided_results

    # --- Unload guide, free VRAM for next precision ---
    unload_guide(guide_model, guide_tok)

print("\n" + "="*65)
print("ALL PRECISION RUNS COMPLETE")
print("="*65)

In [ ]:
# CELL 13 -- Load all results (in case notebook was restarted after runs)

if not precision_results:
    print("Reloading results from disk...")
    for precision in CONFIG["precisions"]:
        fpath = f"{OUTPUT_DIR}/{precision}_guided_results.jsonl"
        if os.path.exists(fpath):
            with open(fpath) as f:
                precision_results[precision] = [json.loads(l) for l in f if l.strip()]
            print(f"  {precision.upper()}: {len(precision_results[precision])} records")
        else:
            print(f"  {precision.upper()}: FILE NOT FOUND — run Cell 12 first")

if not base_results:
    if os.path.exists(BASELINE_FILE):
        with open(BASELINE_FILE) as f:
            base_results = [json.loads(l) for l in f if l.strip()]
        print(f"  BASELINE: {len(base_results)} records")
    else:
        print("  BASELINE: FILE NOT FOUND — run Cell 11 first")

print("Results ready for analysis")

In [ ]:
# CELL 14 -- Per-precision metrics helper

def compute_metrics(results):
    """Compute all three angles for a list of result records."""
    n   = len(results)
    acc = sum(r["correct"] for r in results) / n * 100

    # Angle 2: Vote consistency
    cons       = [r["vote_consistency"] for r in results]
    mean_cons  = np.mean(cons) * 100
    high_cons  = sum(1 for c in cons if c >= 0.8)   # 4 or 5 correct votes
    all_wrong  = sum(1 for c in cons if c == 0.0)

    # Angle 3: ECE
    buckets = [
        (lambda c: c >= 0.80, 0.90),
        (lambda c: 0.60 <= c < 0.80, 0.70),
        (lambda c: 0.40 <= c < 0.60, 0.50),
        (lambda c: c < 0.40, 0.25),
    ]
    ece = 0.0
    for cond, mid in buckets:
        subset = [r for r in results if cond(r["confidence"])]
        if subset:
            bucket_acc = sum(r["correct"] for r in subset) / len(subset)
            ece += (len(subset) / n) * abs(bucket_acc - mid)

    false_conf = sum(1 for r in results if r["confidence"] >= 0.80 and not r["correct"])

    # Latency (guided runs have inference_time; baseline does not)
    times = [r.get("inference_time", 0) for r in results if r.get("inference_time", 0) > 0]
    avg_time = np.mean(times) if times else 0.0

    return {
        "n"          : n,
        "accuracy"   : round(acc, 2),
        "mean_cons"  : round(mean_cons, 2),
        "high_cons"  : high_cons,
        "all_wrong"  : all_wrong,
        "ece"        : round(ece, 4),
        "false_conf" : false_conf,
        "avg_time"   : round(avg_time, 2),
    }


print("Metrics helper ready")

In [ ]:
# CELL 15 -- PRECISION COMPARISON TABLE (main output for the paper)

b_m = compute_metrics(base_results)

# Collect metrics for each precision
rows = []
ref_acc = None   # BF16 accuracy used as reference delta

for precision in CONFIG["precisions"]:
    if precision not in precision_results:
        print(f"  MISSING: {precision} — skipping")
        continue
    m = compute_metrics(precision_results[precision])
    if ref_acc is None:
        ref_acc = m["accuracy"]  # BF16 is first
    delta = round(m["accuracy"] - ref_acc, 2)
    rows.append((precision.upper(), m, delta))

print("=" * 80)
print("  SVAMP — GuidedVote PRECISION COMPARISON  (n={}, seed={})".format(
    len(test_data), CONFIG["random_seed"]))
print("  Solver fixed at FP16. Guide precision varies.")
print("=" * 80)

# Header
print(f"  {'Condition':<12} | {'Accuracy':>9} | {'vs BF16':>8} | {'Vote Cons':>10} | {'ECE':>7} | {'FalseConf':>9} | {'Avg Time':>9}")
print(f"  {'-'*12}-+-{'-'*9}-+-{'-'*8}-+-{'-'*10}-+-{'-'*7}-+-{'-'*9}-+-{'-'*9}")

# Baseline row
print(f"  {'BASELINE':<12} | {b_m['accuracy']:>8.1f}% | {'--':>8} | {b_m['mean_cons']:>9.1f}% | {b_m['ece']:>7.4f} | {b_m['false_conf']:>9} | {'--':>9}")
print(f"  {'-'*12}-+-{'-'*9}-+-{'-'*8}-+-{'-'*10}-+-{'-'*7}-+-{'-'*9}-+-{'-'*9}")

# Precision rows
for prec, m, delta in rows:
    delta_str = (f"+{delta:.1f}" if delta > 0 else f"{delta:.1f}") if delta != 0.0 else "ref"
    print(f"  {prec:<12} | {m['accuracy']:>8.1f}% | {delta_str:>8} | {m['mean_cons']:>9.1f}% | {m['ece']:>7.4f} | {m['false_conf']:>9} | {m['avg_time']:>8.1f}s")

print()
print("  Columns:")
print("    Accuracy   = final answer correct / total questions")
print("    vs BF16    = accuracy delta relative to reference (BF16 = ref)")
print("    Vote Cons  = mean fraction of 5 votes that matched GT (higher = more reliable)")
print("    ECE        = Expected Calibration Error (lower = better calibrated)")
print("    FalseConf  = questions where model was confident (>=80%) AND wrong")
print("    Avg Time   = average seconds per question (guided inference)")

In [ ]:
# CELL 16 -- Per-precision three-angle breakdown (detailed)

for precision in CONFIG["precisions"]:
    if precision not in precision_results:
        continue
    res = precision_results[precision]
    m   = compute_metrics(res)
    b_m = compute_metrics(base_results)

    print(f"\n{'='*65}")
    print(f"  {precision.upper()} — Three-Angle Detail")
    print(f"{'='*65}")

    # Angle 1: Accuracy + compute
    G = CONFIG["guide_params_B"]
    S = CONFIG["solver_params_B"]
    N = CONFIG["n_votes"]
    guided_compute   = G + S * N
    baseline_compute = S * N
    upper_compute    = G * N
    savings_pct      = (1 - guided_compute / upper_compute) * 100

    print(f"\n  ANGLE 1 — Compute Efficiency")
    print(f"    Guided accuracy   : {m['accuracy']:.1f}%")
    print(f"    Baseline accuracy : {b_m['accuracy']:.1f}%")
    print(f"    Gain over baseline: +{m['accuracy'] - b_m['accuracy']:.1f} pp")
    print(f"    Compute           : {guided_compute:.1f}B param-passes ({savings_pct:.0f}% below ceiling)")
    print(f"    Avg time/question : {m['avg_time']:.1f}s")

    # Angle 2: Vote consistency
    print(f"\n  ANGLE 2 — Vote Consistency")
    print(f"    Guided mean vote cons  : {m['mean_cons']:.1f}%")
    print(f"    Baseline mean vote cons: {b_m['mean_cons']:.1f}%")
    print(f"    High-agreement (>=80%) : {m['high_cons']} questions")
    print(f"    All-wrong (0%)         : {m['all_wrong']} questions")

    # Angle 3: Calibration
    print(f"\n  ANGLE 3 — Calibration")
    print(f"    Guided ECE   : {m['ece']:.4f}")
    print(f"    Baseline ECE : {b_m['ece']:.4f}")
    ece_imp = (b_m['ece'] - m['ece']) / max(b_m['ece'], 1e-6) * 100
    print(f"    ECE improvement vs baseline: {ece_imp:.1f}%")
    print(f"    False confidence : {m['false_conf']} questions")

In [ ]:
# CELL 17 -- Save final comparison report to JSON

b_m = compute_metrics(base_results)

final_report = {
    "dataset"    : "SVAMP",
    "n_questions": len(test_data),
    "seed"       : CONFIG["random_seed"],
    "solver"     : {"model": CONFIG["response_model"], "precision": "fp16"},
    "guide_base" : CONFIG["guide_base"],
    "baseline"   : b_m,
    "precisions" : {},
}

ref_acc = None
for precision in CONFIG["precisions"]:
    if precision not in precision_results:
        continue
    m = compute_metrics(precision_results[precision])
    if ref_acc is None:
        ref_acc = m["accuracy"]
    m["delta_vs_bf16"] = round(m["accuracy"] - ref_acc, 2)
    m["gain_vs_baseline"] = round(m["accuracy"] - b_m["accuracy"], 2)
    final_report["precisions"][precision] = m

report_path = f"{OUTPUT_DIR}/precision_comparison_report.json"
with open(report_path, "w") as f:
    json.dump(final_report, f, indent=2)

print(f"✅ Final report saved to: {report_path}")
print("\nCommit this notebook to preserve all outputs.")